In [2]:
import instructor
import json
from litellm import completion
from dotenv import load_dotenv

from prompt_strategy.prompts import SYSTEM_PROMPT
from text_to_json.schema_design import DomainStory

from pydantic import BaseModel, Field
from utils.api_request import api_response

load_dotenv()

True

In [3]:
alphorn_5_path = r"/promptfoo-eval/alphorn_text/alphorn-5.txt"


def load_content(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


content = load_content(alphorn_5_path)

In [6]:
def one_phase_zeroshot(model_name):
    messages_1 = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": content},
    ]

    resp = api_response(model_name=model_name, messages=messages_1, schema=DomainStory)
    print("type of resp: ", type(resp))

    resp_json = resp.model_dump(mode="json")
    print("type of resp_json: ", type(resp_json))
    print("resp_json: ", resp_json)


one_phase_zeroshot("gpt-4o-mini")



type <class 'text_to_json.schema_design.DomainStory'>
type of resp:  <class 'text_to_json.schema_design.DomainStory'>
type of resp_json:  <class 'dict'>
resp_json:  {'title': 'Online Leasing Service Process', 'actors': [{'id': 'online_leasing_service', 'name': 'Online Leasing Service', 'type': 'System'}, {'id': 'rating_agency_website', 'name': 'Rating Agency Website', 'type': 'System'}, {'id': 'risk_manager', 'name': 'Risk Manager', 'type': 'Person'}], 'work_objects': [{'id': 'contract', 'name': 'Contract', 'description': 'contract', 'instances': [{'instance_id': 'contract_1', 'note': None}], 'icon': None}, {'id': 'credit_rating_report', 'name': 'Credit Rating Report', 'description': 'report', 'instances': [{'instance_id': 'credit_rating_report_1', 'note': None}], 'icon': None}, {'id': 'risk_assessment', 'name': 'Risk Assessment', 'description': 'assessment', 'instances': [{'instance_id': 'risk_assessment_1', 'note': None}], 'icon': None}], 'activities': [{'step': 1, 'text': None, 'mai

In [ ]:
request = completion(
    # or gemini/ claude sonnet
    model="gpt-5.4",
    messages=messages,
    response_format=DomainStory
)

In [11]:
one_phase_zeroshot("gpt-5.4")

saved to new file


In [1]:
few_shot_example_1 = """
1.The customer reports damage to the leased vehicle.

2.The leasing company registers the damage report.

3. The company checks the contract conditions.

4. The company creates a repair order.
Note: when the damage is covered

5. Company informs the customer about the next steps.
"""

In [3]:
def two_phase_zeroshot(model_name: str, content: str):
    # phase 1, free formf extraction
    prompt_1 = f"""Identify the Actors, Work Objects, and chronological Activities in the following text.
    Let's think step by step.

    Text: {content}"""

    text_1 = """

    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_1},
    ]

    resp1 = api_response(model_name=model_name, messages=messages)
    print("resp1: \n", resp1)

    # phase 2
    promt_2 = """Organize the previous extracted content to the predefined  schema DomainStory.
    Make sure to present the entities precisely in the same words as in the original paragraph.
    Let's think step by step ."""

    messages.append({"role": "assistant", "content": resp1})
    messages.append({"role": "user", "content": promt_2})

    resp2 = api_response(model_name=model_name, messages=messages, schema=DomainStory)
    print("resp2: \n", resp2)

    output_obj = DomainStory.model_validate_json(resp2)

    output_path = r"C:\code\NL_2_DST\instructor\output\gpt5.4-2phase-jsonoutputfirstphase.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output_obj.model_dump(mode="json"), f, ensure_ascii=False, indent=2)

    print("saved to new file")




In [9]:
two_phase_zeroshot("gpt-5.4", few_shot_example_1)

resp1: 
 {
  "actors": [
    "customer",
    "leasing company"
  ],
  "work_objects": [
    "damage",
    "leased vehicle",
    "damage report",
    "contract conditions",
    "repair order",
    "next steps"
  ],
  "activities": [
    {
      "step": 1,
      "actor": "customer",
      "action": "reports",
      "work_object": "damage",
      "recipient": "leasing company"
    },
    {
      "step": 2,
      "actor": "leasing company",
      "action": "registers",
      "work_object": "damage report"
    },
    {
      "step": 3,
      "actor": "leasing company",
      "action": "checks",
      "work_object": "contract conditions"
    },
    {
      "step": 4,
      "actor": "leasing company",
      "action": "creates",
      "work_object": "repair order",
      "condition": "when the damage is covered"
    },
    {
      "step": 5,
      "actor": "leasing company",
      "action": "informs",
      "work_object": "next steps",
      "recipient": "customer"
    }
  ]
}
resp2: 
 {"title

In [ ]:
def one_phase_fewshot(model_name):
    messages_1 = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": content},
    ]

    resp = api_response(model_name=model_name, messages=messages_1, schema=DomainStory)

    output_obj = DomainStory.model_validate_json(resp)

    output_path = r"C:\code\NL_2_DST\instructor\output\gpt5.4-one_phase.bjson"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output_obj.model_dump(mode="json"), f, ensure_ascii=False, indent=2)

    print("saved to new file")




In [ ]:
def two_phase_fewshot(model_name: str, content: str):
    # phase 1, free formf extraction
    prompt_1 = f"""Identify the Actors, Work Objects, and chronological Activities in the following text.
    Let's think step by step.

    Text: {content}"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_1},
    ]

    resp1 = api_response(model_name=model_name, messages=messages)
    print("resp1: \n", resp1)

    # phase 2
    promt_2 = """Organize the previous extracted content to the predefined  schema DomainStory.
    Make sure to present the entities precisely in the same words as in the original paragraph.
    Let's think step by step ."""

    messages.append({"role": "assistant", "content": resp1})
    messages.append({"role": "user", "content": promt_2})

    resp2 = api_response(model_name=model_name, messages=messages, schema=DomainStory)
    print("resp2: \n", resp2)

    output_obj = DomainStory.model_validate_json(resp2)

    output_path = r"C:\code\NL_2_DST\instructor\output\gpt5.4-2phase-jsonoutputfirstphase.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output_obj.model_dump(mode="json"), f, ensure_ascii=False, indent=2)

    print("saved to new file")




In [6]:
from pydantic import BaseModel

messages = [{"role": "user", "content": "List 5 important events in the XIX century"}]


class CalendarEvent(BaseModel):
    name: str
    id: str = Field(
        description="Unique identifier for the actor, usually the same as the name but connected through underscore."
    )
    date: str
    participants: list[str]


class EventsList(BaseModel):
    events: list[CalendarEvent]


resp = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
    response_format=EventsList
)

print("Received={}".format(resp))

events_list = EventsList.model_validate_json(resp.choices[0].message.content)

Received=ModelResponse(id='chatcmpl-DrIOdJTU6wBGMvs5JMymOsJ2pZ3u3', created=1781594471, model='gpt-4o-2024-08-06', object='chat.completion', system_fingerprint='fp_0811fbd893', choices=[Choices(finish_reason='stop', index=0, message=Message(content='{"events":[{"name":"Congress of Vienna","id":"congress_of_vienna","date":"1814-1815","participants":["Austria","Britain","France","Russia","Prussia"]},{"name":"American Civil War","id":"american_civil_war","date":"1861-1865","participants":["Union","Confederate States"]},{"name":"Abolition of Slavery in the U.S.","id":"abolition_of_slavery_in_us","date":"1865","participants":["United States"]},{"name":"Revolutions of 1848","id":"revolutions_of_1848","date":"1848","participants":["France","Italy","German States","Hungary","Austria"]},{"name":"First Industrial Revolution","id":"first_industrial_revolution","date":"1760-1840","participants":["Britain","Europe","United States"]}]}', role='assistant', tool_calls=None, function_call=None, provide

In [ ]:
gpt = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "capital_response",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "country": {"type": "string"},
                    "capital": {"type": "string"}
                },
                "required": ["country", "capital"],
                "additionalProperties": False
            }
        }
    }
)

In [14]:


gemini = completion(
    model="gemini/gemini-3.1-pro-preview",
    messages=[{"role": "user", "content": "write code for saying hi from LiteLLM"}],
    reasoning_effort="disable"
)



In [15]:
gemini

ModelResponse(id='9AAwapqoE7bqxN8P0c2TMQ', created=1781530866, model='gemini-3.1-pro-preview', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='Here is the Python code to say "hi" using **LiteLLM**. \n\nLiteLLM allows you to call over 100+ LLM APIs using the standard OpenAI format. In this example, we\'ll use OpenAI\'s `gpt-3.5-turbo`, but you can easily swap it out for Anthropic, Google, HuggingFace, etc.\n\n### Prerequisites\nFirst, install the package via pip:\n```bash\npip install litellm\n```\n\n### Python Code\n```python\nimport os\nfrom litellm import completion\n\n# 1. Set your API key (replace with your actual API key)\n# If you want to use Anthropic or Gemini, use ANTHROPIC_API_KEY or GEMINI_API_KEY instead\nos.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"\n\n# 2. Call the model\nresponse = completion(\n    model="gpt-3.5-turbo", # You can change this to "claude-3-haiku-20240307", "gemini-1.5-